# Empty-Car Hunting in Rail Trucks

This MxV Rail capstone notebook explores how wheel and rail profile measurements relate to empty-car hunting. It compares logistic regression and XGBoost models, then checks how candidate flange measurement thresholds relate to observed hunting rates.

## Exploratory Data Analysis (EDA)

Explore hunting rates by rail profile and compare wheel measurements across hunting outcomes. Prepare the features and target used to start modeling.

In [ ]:
# Shared input locations; see INSTALL.md for the Teams download layout.
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository; see INSTALL.md.")
sys.path.insert(0, str(ROOT / "src"))
from project_paths import data_directory, find_project_root, merged_file
ROOT = find_project_root(ROOT)
DATA = data_directory(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the supplied merged dataset from the shared data folder
final_data = pd.read_csv(merged_file(DATA))

# Check that it loaded correctly
final_data.shape  

In [ ]:
# Compare how often hunting occurs across the seven rail profiles in percentage
rail_hunting = final_data.groupby("Rail_Profile")["Empty_Truck_Hunted"].mean() * 100

print(rail_hunting)

Profiles 1, 2 and 4 have the highest hunting fractions in this
dataset. These are repeated assessments of the same trucks under seven rails,
so the bar heights describe the supplied simulations rather than independent
rail samples.

In [ ]:
# Visualize differences in hunting frequency across rail profiles
rail_hunting.plot(kind="bar")

plt.xlabel("Rail Profile")
plt.ylabel("Hunting Rate (%)")
plt.title("Empty-Car Hunting Rate by Rail Profile")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Compare equivalent conicity between hunting and non-hunting cases
final_data.boxplot(
    column="Eq Con 3mm",
    by="Empty_Truck_Hunted"
)

plt.xlabel("Truck Hunted")
plt.ylabel("Equivalent Conicity at 3 mm")
plt.title("Equivalent Conicity vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

**Suggest a higher eq con is associated with empty car hunting since the median is noticeably higher on the true side. The wider range suggest hunting occurs on a broader range of eq con values.**

In [ ]:
# Compare average equivalent conicity by rail profile and hunting outcome
conicity_by_profile = final_data.groupby(
    ["Rail_Profile", "Empty_Truck_Hunted"]
)["Eq Con 3mm"].mean().unstack()

conicity_by_profile.plot(kind="bar")

plt.xlabel("Rail Profile")
plt.ylabel("Average Equivalent Conicity at 3 mm")
plt.title("Equivalent Conicity by Rail Profile and Hunting Outcome")
plt.xticks(rotation=0)
plt.legend(["No Hunting", "Hunting"])
plt.show()

**Equivalent conicity was generally higher for hunting cases across most rail profiles. Rail profiles 5 and 6 were exceptions, with higher average equivalent conicity among non-hunting cases. These two profiles also had the lowest overall hunting rates. However, the sample sizes within each rail profile and hunting group are relatively small, so these patterns should be treated as exploratory rather than conclusive. The results suggest that the relationship between equivalent conicity and hunting may depend on the rail profile and should be investigated further.**

In [ ]:
# Create wheelset hollow wear features from the left and right wheels
final_data["Hollow_Wear_Avg"] = (
    final_data["Max Hollow Wear - AAR_L"] +
    final_data["Max Hollow Wear - AAR_R"]
) / 2

final_data["Hollow_Wear_Diff"] = abs(
    final_data["Max Hollow Wear - AAR_L"] -
    final_data["Max Hollow Wear - AAR_R"]
)

In [ ]:
# Compare wheelset hollow wear between hunting and non-hunting cases
final_data.boxplot(
    column="Hollow_Wear_Avg",
    by="Empty_Truck_Hunted"
)

plt.xlabel("Truck Hunted")
plt.ylabel("Average Hollow Wear")
plt.title("Wheelset Hollow Wear vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

In [ ]:
# Compare hollow wear while hiding extreme points so the main distributions are visible
final_data.boxplot(
    column="Hollow_Wear_Avg",
    by="Empty_Truck_Hunted",
    showfliers=False
)

plt.xlabel("Truck Hunted")
plt.ylabel("Average Hollow Wear")
plt.title("Wheelset Hollow Wear vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

Hunting cases have a more negative median for this signed
hollow-wear field, but the distributions overlap substantially. Confirm the
source sign convention before interpreting a more negative value as more wear.
The second plot hides outlier markers; it does not remove rows from the data.

In [ ]:
# Check whether uneven hollow wear across the wheelset is related to hunting
final_data.boxplot(
    column="Hollow_Wear_Diff",
    by="Empty_Truck_Hunted",
    showfliers=False
)

plt.xlabel("Truck Hunted")
plt.ylabel("Left-Right Hollow Wear Difference")
plt.title("Wheelset Hollow Wear Difference vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

**The median left-right hollow wear difference was about the same for hunting and non-hunting cases. The hunting cases were more closely grouped together, while the non-hunting cases had a wider range. Overall, there was a lot of overlap between the two groups, so the left-right hollow wear difference does not seem to have a strong relationship with hunting on its own.**

In [ ]:
# Create wheelset qR features from the left and right wheels
final_data["qR_Avg"] = (
    final_data["qR_L"] +
    final_data["qR_R"]
) / 2

final_data["qR_Diff"] = abs(
    final_data["qR_L"] -
    final_data["qR_R"]
)

In [ ]:
# Compare average wheelset qR between hunting and non-hunting cases
final_data.boxplot(
    column="qR_Avg",
    by="Empty_Truck_Hunted",
    showfliers=False
)

plt.xlabel("Truck Hunted")
plt.ylabel("Average qR")
plt.title("Wheelset qR vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

**Average qR was very similar between hunting and non-hunting cases. The hunting cases had a slightly higher median and a slightly larger spread in the middle of the data, but both groups had about the same overall range. This suggests that average qR does not have a strong relationship with hunting on its own.**

In [ ]:
# Check whether left-right qR differences are related to hunting
final_data.boxplot(
    column="qR_Diff",
    by="Empty_Truck_Hunted",
    showfliers=False
)

plt.xlabel("Truck Hunted")
plt.ylabel("Left-Right qR Difference")
plt.title("Wheelset qR Difference vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

**The median left-right qR difference was about the same for hunting and non-hunting cases. The non-hunting cases had a larger middle spread and a wider overall range. There was a lot of overlap between the groups, so the left-right qR difference does not appear to have a strong relationship with hunting on its own.**

In [ ]:
# Create wheelset flange width features from the left and right wheels
final_data["Flange_Width_Avg"] = (
    final_data["Flange Width_L"] +
    final_data["Flange Width_R"]
) / 2

final_data["Flange_Width_Diff"] = abs(
    final_data["Flange Width_L"] -
    final_data["Flange Width_R"]
)

In [ ]:
# Compare average flange width between hunting and non-hunting cases
final_data.boxplot(
    column="Flange_Width_Avg",
    by="Empty_Truck_Hunted",
    showfliers=False
)

plt.xlabel("Truck Hunted")
plt.ylabel("Average Flange Width")
plt.title("Wheelset Flange Width vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

**Average flange width was higher for hunting cases than for non-hunting cases. The two groups had a similar middle spread, but the hunting group was shifted toward higher flange width values overall. This suggests that flange width may have some relationship with hunting and is worth investigating further.**

In [ ]:
# Check whether left-right flange width differences are related to hunting
final_data.boxplot(
    column="Flange_Width_Diff",
    by="Empty_Truck_Hunted",
    showfliers=False
)

plt.xlabel("Truck Hunted")
plt.ylabel("Left-Right Flange Width Difference")
plt.title("Wheelset Flange Width Difference vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

The left/right flange-width differences have similar medians and
substantial overlap. They do not separate the outcomes clearly on their own.

In [ ]:
# Create wheelset maximum flange angle features
final_data["Flange_Angle_Avg"] = (
    final_data["Max Flange Angle_L"] +
    final_data["Max Flange Angle_R"]
) / 2

final_data["Flange_Angle_Diff"] = abs(
    final_data["Max Flange Angle_L"] -
    final_data["Max Flange Angle_R"]
)

In [ ]:
# Compare maximum flange angle between hunting and non-hunting cases
final_data.boxplot(
    column="Flange_Angle_Avg",
    by="Empty_Truck_Hunted",
    showfliers=False
)

plt.xlabel("Truck Hunted")
plt.ylabel("Average Maximum Flange Angle")
plt.title("Wheelset Maximum Flange Angle vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

**Average maximum flange angle was slightly higher for non-hunting cases. The two groups had similar overall ranges and middle spreads, with the non-hunting group shifted slightly higher. There was still a lot of overlap, so maximum flange angle does not appear to have a strong relationship with hunting on its own.**

In [ ]:
# Check whether left-right maximum flange angle differences are related to hunting
final_data.boxplot(
    column="Flange_Angle_Diff",
    by="Empty_Truck_Hunted",
    showfliers=False
)

plt.xlabel("Truck Hunted")
plt.ylabel("Left-Right Maximum Flange Angle Difference")
plt.title("Wheelset Maximum Flange Angle Difference vs Empty-Car Hunting")
plt.suptitle("")
plt.show()

**The hunting cases had a lower median left-right maximum flange angle difference, but they also had a wider middle spread and a much larger overall range. The non-hunting group was mostly contained within the middle range of the hunting group. This suggests that flange angle difference may be more variable in hunting cases, although it does not clearly separate hunting from non-hunting cases.**

In [ ]:
# Create wheelset averages for the main wheel wear measurements
measurements = [
    "Flange Width (Sd)",
    "Flange Height (Sh)",
    "qR",
    "Flange Width",
    "Max Hollow Wear",
    "Flange Radius",
    "Max Flange Angle",
    "Min Flange Root Radius",
    "Max Hollow Wear - AAR"
]

for measurement in measurements:
    final_data[measurement + "_Avg"] = (
        final_data[measurement + "_L"] +
        final_data[measurement + "_R"]
    ) / 2

In [ ]:
# Calculate effect size to compare hunting and non-hunting groups
effect_sizes = {}

for measurement in measurements:
    column = measurement + "_Avg"

    hunted = final_data[final_data["Empty_Truck_Hunted"] == True][column].dropna()
    not_hunted = final_data[final_data["Empty_Truck_Hunted"] == False][column].dropna()

    mean_diff = hunted.mean() - not_hunted.mean()

    pooled_std = (
        ((len(hunted) - 1) * hunted.var() +
         (len(not_hunted) - 1) * not_hunted.var()) /
        (len(hunted) + len(not_hunted) - 2)
    ) ** 0.5

    effect_sizes[measurement] = mean_diff / pooled_std

effect_sizes = pd.Series(effect_sizes)
effect_sizes = effect_sizes.reindex(effect_sizes.abs().sort_values(ascending=False).index)

print(effect_sizes)

Cohen's d describes the difference in group means relative to
their pooled spread. Positive values mean a larger average among hunting rows.
Flange width has the largest positive difference here, with smaller shifts for
qR and Sd. These are descriptive effect sizes from repeated axle/rail rows,
not significance tests or counts of independent vehicles.

In [ ]:
# Check whether the truck result matches the minimum result from its two axles
axle_result = final_data.groupby(
    ["Car", "Truck", "Rail_Profile"]
)["Empty Truck from Acc HS [mph]"].min().reset_index()

truck_result = final_data[
    ["Car", "Truck", "Rail_Profile", "Min Empty Truck from Acc HS [mph]"]
].drop_duplicates()

check = axle_result.merge(
    truck_result,
    on=["Car", "Truck", "Rail_Profile"]
)

matches = (
    check["Empty Truck from Acc HS [mph]"] ==
    check["Min Empty Truck from Acc HS [mph]"]
)

print(matches.value_counts())

In [ ]:
# Classify empty-car hunting using the truck-level results
final_data["Truck_Hunted"] = (
    final_data["Min Empty Truck from Acc HS [mph]"] < 80
)

# Check whether both axles agree with the truck-level classification
print(
    final_data.groupby(["Car", "Truck", "Rail_Profile"])["Truck_Hunted"]
    .nunique().value_counts()
)

In [ ]:
# Select physical wheel characteristics and rail profile
features = [
    "Flange_Width_Avg",
    "Flange Width (Sd)_Avg",
    "qR_Avg",
    "Hollow_Wear_Avg",
    "Flange_Angle_Avg",
    "Rail_Profile"
]

X = final_data[features].copy()
y = final_data["Truck_Hunted"].astype(int)

print("Features:", X.columns.tolist())
print("Missing values:", X.isna().sum().sum())
print("Target classes:", sorted(y.unique()))

## Logistic Regression

Split the data by car, compare wheel feature sets, and tune L2 regularization using grouped cross-validation.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Keep all data from the same car together to avoid data leakage
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=final_data["Car"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training classes:", y_train.value_counts().to_dict())
print("Testing classes:", y_test.value_counts().to_dict())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = [
    "Flange_Width_Avg",
    "Flange Width (Sd)_Avg",
    "qR_Avg",
    "Hollow_Wear_Avg",
    "Flange_Angle_Avg"
]

categorical_features = ["Rail_Profile"]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)



In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Predict hunting on unseen cars
y_pred = model.predict(X_test)

# Evaluate predictions
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Hunting", "Hunting"],
    zero_division=0
))

# Visualize correct and incorrect predictions
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No Hunting", "Hunting"],
    cmap="Blues"
)

plt.title("Logistic Regression - Test Results")
plt.show()

The confusion matrix shows much higher hunting recall than
no-hunting recall. The outcome imbalance makes accuracy look better than the
ability to distinguish both classes. The later balanced models and balanced-
accuracy scores help check that trade-off.

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

coefficients = model.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df = coef_df.sort_values("Coefficient")

coef_df.plot(
    x="Feature",
    y="Coefficient",
    kind="barh",
    legend=False,
    figsize=(9, 6)
)

plt.axvline(0, color="black")
plt.xlabel("Logistic Regression Coefficient")
plt.title("Features Associated with Hunting")
plt.tight_layout()
plt.show()

In [ ]:
# Add equivalent conicity to the existing features
X2 = final_data[features + ["Eq Con 3mm"]].copy()

# Use the same train/test split as Model 1
X2_train = X2.iloc[train_idx]
X2_test = X2.iloc[test_idx]

numeric_features_2 = numeric_features + ["Eq Con 3mm"]

preprocessor_2 = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features_2),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

model_2 = Pipeline([
    ("preprocessor", preprocessor_2),
    ("classifier", LogisticRegression(max_iter=1000))
])

model_2.fit(X2_train, y_train)



In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Predictions from both models
y_pred_1 = model.predict(X_test)
y_pred_2 = model_2.predict(X2_test)

print("MODEL 1 — Without Equivalent Conicity")
print("Accuracy:", round(accuracy_score(y_test, y_pred_1), 3))
print(classification_report(y_test, y_pred_1, zero_division=0))

print("\nMODEL 2 — With Equivalent Conicity")
print("Accuracy:", round(accuracy_score(y_test, y_pred_2), 3))
print(classification_report(y_test, y_pred_2, zero_division=0))

**MODEL 1: Left Wheel Only**

In [ ]:
wheel_features = [
    "Flange Width",
    "Flange Width (Sd)",
    "Flange Height (Sh)",
    "qR",
    "Max Hollow Wear - AAR",
    "Max Flange Angle",
    "Flange Radius",
    "Min Flange Root Radius"
]

for feature in wheel_features:
    print(feature)
    print("Left:", feature + "_L" in final_data.columns)
    print("Right:", feature + "_R" in final_data.columns)
    print()

In [ ]:
# Select the left wheel measurements
left_features = [feature + "_L" for feature in wheel_features]

# Set up the features and hunting outcome
X_left = final_data[left_features].copy()
y = final_data["Truck_Hunted"].astype(int)

# Check the data before modeling
print("X shape:", X_left.shape)
print("Missing values:", X_left.isna().sum().sum())
print("Hunting outcomes:")
print(y.value_counts())

In [ ]:


# Split by car so the same car isn't in both sets
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X_left, y, groups=final_data["Car"])
)

# Create the training and testing sets
X_left_train = X_left.iloc[train_idx]
X_left_test = X_left.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Check the sizes of each set
print("Training observations:", len(X_left_train))
print("Testing observations:", len(X_left_test))
print("Training hunting outcomes:")
print(y_train.value_counts())
print("Testing hunting outcomes:")
print(y_test.value_counts())

In [ ]:
# Scale the wheel measurements and build the model
left_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train the model using the left wheel data
left_model.fit(X_left_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix
# Predict hunting on the testing data
y_pred_left = left_model.predict(X_left_test)

# Calculate accuracy and display performance metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred_left), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_left))

# Show the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_left))

**MODEL 2: RIGHT WHEEL ONLY**

In [ ]:
# Select the right wheel measurements
right_features = [feature + "_R" for feature in wheel_features]
X_right = final_data[right_features].copy()

# Use the same training and testing observations
X_right_train = X_right.iloc[train_idx]
X_right_test = X_right.iloc[test_idx]

# Build and train the right wheel model
right_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

right_model.fit(X_right_train, y_train)


In [ ]:
# Predict hunting using the right wheel measurements
y_pred_right = right_model.predict(X_right_test)

# Display accuracy and classification metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred_right), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_right))

# Show the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_right))

**MODEL 3: LEFT/RIGHT WEAR DIFFERENCE**

In [ ]:
# Calculate differences between left and right wheel measurements
difference_features = []

for feature in wheel_features:
    column_name = feature + "_Diff"

    final_data[column_name] = abs(
        final_data[feature + "_L"] - final_data[feature + "_R"]
    )

    difference_features.append(column_name)

# Set up the difference-only dataset
X_diff = final_data[difference_features].copy()

# Check the features and missing values
print("Features:", difference_features)
print("X shape:", X_diff.shape)
print("Missing values:", X_diff.isna().sum().sum())

In [ ]:
# Use the same training and testing split as the previous models
X_diff_train = X_diff.iloc[train_idx]
X_diff_test = X_diff.iloc[test_idx]

# Build the logistic regression model using wheel differences
diff_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train the model
diff_model.fit(X_diff_train, y_train)


In [ ]:
# Predict hunting using left/right wheel differences
y_pred_diff = diff_model.predict(X_diff_test)

# Display accuracy and classification metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred_diff), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_diff))

# Show the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_diff))

**MODEL 4: BOTH WHEELS COMBINED**

In [ ]:
# Combine left and right wheel measurements
combined_features = left_features + right_features

X_combined = final_data[combined_features].copy()

# Use the same training and testing split
X_combined_train = X_combined.iloc[train_idx]
X_combined_test = X_combined.iloc[test_idx]

# Check the combined dataset
print("X shape:", X_combined.shape)
print("Missing values:", X_combined.isna().sum().sum())

In [ ]:
# Build the logistic regression model using both wheels
combined_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train the model using both left and right wheel measurements
combined_model.fit(X_combined_train, y_train)

In [ ]:
# Predict hunting using both wheels
y_pred_combined = combined_model.predict(X_combined_test)

# Display accuracy and classification metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred_combined), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_combined))

# Show the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_combined))

**MODEL 5: WHEEL AVERAGES + DIFFERENCES**

In [ ]:
# Calculate average measurements across both wheels
average_features = []

for feature in wheel_features:
    column_name = feature + "_Avg"

    final_data[column_name] = (
        final_data[feature + "_L"] + final_data[feature + "_R"]
    ) / 2

    average_features.append(column_name)

# Combine wheel averages and left/right differences
avg_diff_features = average_features + difference_features

X_avg_diff = final_data[avg_diff_features].copy()

# Use the same training and testing observations
X_avg_diff_train = X_avg_diff.iloc[train_idx]
X_avg_diff_test = X_avg_diff.iloc[test_idx]

# Check the dataset
print("X shape:", X_avg_diff.shape)
print("Missing values:", X_avg_diff.isna().sum().sum())

In [ ]:
# Build the logistic regression model using averages and differences
avg_diff_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train the model
avg_diff_model.fit(X_avg_diff_train, y_train)

In [ ]:
# Predict hunting using wheel averages and differences
y_pred_avg_diff = avg_diff_model.predict(X_avg_diff_test)

# Display accuracy and classification metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred_avg_diff), 3))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_avg_diff))

# Show the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_avg_diff))

**BALANCED AVERAGES + DIFFERENCES MODEL**

In [ ]:
# Build logistic regression with balanced class weights
balanced_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Train using wheel averages and differences
balanced_model.fit(X_avg_diff_train, y_train)


In [ ]:
from sklearn.metrics import balanced_accuracy_score

# Predict hunting using balanced logistic regression
y_pred_balanced = balanced_model.predict(X_avg_diff_test)

# Display model performance
print("Accuracy:", round(accuracy_score(y_test, y_pred_balanced), 3))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_balanced), 3))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced))

# Show the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

**BALANCED LEFT WHEEL**

In [ ]:
# Build balanced logistic regression using left wheel measurements
balanced_left_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Train using left wheel measurements
balanced_left_model.fit(X_left_train, y_train)


In [ ]:
# Predict hunting using the balanced left wheel model
y_pred_balanced_left = balanced_left_model.predict(X_left_test)

# Display accuracy and classification metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred_balanced_left), 3))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_balanced_left), 3))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced_left))

# Show the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced_left))

**FEATURE IMPORTANCE OF AVERAGES AND DIFFERENCES MODEL**

In [ ]:
# Extract coefficients from the balanced logistic regression model
coefficients = balanced_model.named_steps["classifier"].coef_[0]

# Create a table of wheel features and their coefficients
feature_importance = pd.DataFrame({
    "Feature": avg_diff_features,
    "Coefficient": coefficients,
    "Absolute Coefficient": abs(coefficients)
})

# Sort by coefficient magnitude
feature_importance = feature_importance.sort_values(
    "Absolute Coefficient", ascending=False
)

print(feature_importance.to_string(index=False))

## Reading the balanced logistic-regression coefficients

Flange-height difference has the largest absolute coefficient in this fit.
Larger values increase its predicted hunting score; the signs for minimum
flange-root-radius difference and mean AAR hollow wear are negative. These are
conditional model associations, not effects of changing a wheel measurement.

The held-car test has modest balanced accuracy. The correlation table below
shows why individual coefficients need care: flange-height difference overlaps
strongly with hollow-wear and flange-root-radius summaries. The grouped
cross-validation results give a broader check than this single test split.

In [ ]:
# Select the six strongest features from our model
top_features = feature_importance["Feature"].head(6).tolist()

# Check whether these measurements are correlated
correlations = X_avg_diff_train[top_features].corr()

print(correlations.round(2).to_string())

Flange-height difference is strongly correlated with hollow-wear
difference, average hollow wear and flange-root-radius difference in the
training data. Flange width is less closely tied to those summaries. This
helps explain why coefficient magnitude is not a reliable ranking of
independent physical mechanisms.

**Grouped cross-validation to see whether the flange-related findings hold across different cars**

In [ ]:
from sklearn.model_selection import GroupKFold, cross_validate

# Keep all observations from the same car together
cv = GroupKFold(n_splits=5)

# Evaluate balanced logistic regression across five folds
cv_results = cross_validate(
    balanced_model,
    X_avg_diff,
    y,
    groups=final_data["Car"],
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "f1_macro": "f1_macro"
    }
)

# Display average performance across folds
for metric in ["accuracy", "balanced_accuracy", "f1_macro"]:
    scores = cv_results["test_" + metric]
    print(f"{metric}: {scores.mean():.3f} (+/- {scores.std():.3f})")

## Regularization check

Grouped cross-validation shows variation between held-car folds. The search
below chooses regularization using those same folds; its best CV score is a
tuning result, not an additional independent test.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Build balanced logistic regression with L2 regularization
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        penalty="l2"
    ))
])

# Test different regularization strengths
param_grid = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

# Tune using grouped cross-validation
ridge_search = GridSearchCV(
    ridge_model,
    param_grid,
    cv=GroupKFold(n_splits=5),
    scoring="balanced_accuracy"
)

ridge_search.fit(
    X_avg_diff,
    y,
    groups=final_data["Car"]
)

print("Best C:", ridge_search.best_params_["classifier__C"])
print("Best CV Balanced Accuracy:", round(ridge_search.best_score_, 3))

L2 tuning helped a little. Balanced accuracy went from 0.552 to 0.572 in cross-validation grouped by car.

## XGBoost Modeling and Validation

Train and evaluate XGBoost, use SHAP to interpret predictions, and check candidate flange thresholds against observed hunting rates.

**Step 1: Train and tune XGBoost**

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV

#Build an XGBoost model using all wheel averages and differences
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=1
)

#Test a small range of model settings
param_grid = {
    "max_depth": [2, 3],
    "learning_rate": [0.03, 0.1],
    "n_estimators": [100, 200],
    "min_child_weight": [3, 8]
}

#Tune the model while keeping observations from the same car together
xgb_search = GridSearchCV(
    xgb_model,
    param_grid,
    cv=GroupKFold(n_splits=5),
    scoring="balanced_accuracy",
    n_jobs=-1
)

xgb_search.fit(
    X_avg_diff_train,
    y_train,
    groups=final_data["Car"].iloc[train_idx]
)

print("Best parameters:", xgb_search.best_params_)
print("Best training CV balanced accuracy:", round(xgb_search.best_score_, 3))

In [ ]:
# Use the best XGBoost model from cross validation
best_xgb = xgb_search.best_estimator_

# Predict hunting on the held out test cars
y_pred_xgb = best_xgb.predict(X_avg_diff_test)

# Display model performance
print("Accuracy:", round(accuracy_score(y_test, y_pred_xgb), 3))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_xgb), 3))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

On the test cars, XGBoost got 0.610 balanced accuracy. It caught 82% of the hunting cases and correctly identified 39% of the cases that did not hunt.

In [ ]:
import shap
import matplotlib.pyplot as plt

#Explain predictions on the held out test cars
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer(X_avg_diff_test)

#Show which measurements push predictions toward or away from hunting
shap.plots.beeswarm(shap_values, max_display=16)
plt.show()

In [ ]:
#Look at how average flange width affects hunting predictions
shap.plots.scatter(
    shap_values[:, "Flange Width_Avg"]
)

In [ ]:
#Look at how left/right flange height differences affect predictions
shap.plots.scatter(
    shap_values[:, "Flange Height (Sh)_Diff"]
)

The first SHAP plot ranks the measurements by how much they move the model’s predictions. The next two zoom in on average flange width and the difference in flange height between the two wheels. Each dot is a test observation. Positive SHAP values push the model toward predicting hunting, and negative values push it the other way.

In [ ]:
# Focus on the main observed measurement ranges
width_vals = np.linspace(28, 35, 50)
height_vals = np.linspace(0, 2.5, 50)

W, H = np.meshgrid(width_vals, height_vals)
Z = np.zeros_like(W)

for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        X_temp = X_avg_diff_train.copy()
        X_temp["Flange Width_Avg"] = W[i, j]
        X_temp["Flange Height (Sh)_Diff"] = H[i, j]

        Z[i, j] = best_xgb.predict_proba(X_temp)[:, 1].mean()

In [ ]:
# Find the combination with the highest predicted hunting probability
max_idx = np.unravel_index(np.argmax(Z), Z.shape)

best_width = W[max_idx]
best_height = H[max_idx]
max_prob = Z[max_idx]

print("One grid maximum; the plateau can contain many combinations.")
print(f"Average Flange Width: {best_width:.3f}")
print(f"Flange Height Difference: {best_height:.3f}")
print(f"Highest Predicted Hunting Probability: {max_prob:.2%}")

In [ ]:
# Find combinations within 0.5 percentage points of the maximum
high_risk = Z >= Z.max() - 0.005

print("Highest predicted probability:", round(Z.max() * 100, 2), "%")

print("Flange Width Range:",
      round(W[high_risk].min(), 3),
      "to",
      round(W[high_risk].max(), 3))

print("Flange Height Difference Range:",
      round(H[high_risk].min(), 3),
      "to",
      round(H[high_risk].max(), 3))

In [ ]:
# Compare observed hunting rates above and below the model threshold
threshold = 1.327

comparison = final_data.copy()

comparison["Height Group"] = np.where(
    comparison["Flange Height (Sh)_Diff"] >= threshold,
    "High Difference",
    "Low Difference"
)

results = comparison.groupby("Height Group")["Truck_Hunted"].agg(
    Count="count",
    Hunting_Rate="mean"
)

results["Hunting_Rate"] *= 100

print(results.round(2))

In [ ]:
# Compare observed hunting rates for both flange measurements
comparison = final_data.copy()

comparison["Width Group"] = np.where(
    comparison["Flange Width_Avg"] >= 32.5,
    "High Width", "Low Width"
)

comparison["Height Group"] = np.where(
    comparison["Flange Height (Sh)_Diff"] >= 1.327,
    "High Difference", "Low Difference"
)

results = comparison.groupby(
    ["Width Group", "Height Group"]
)["Truck_Hunted"].agg(
    Count="count",
    Hunting_Rate="mean"
)

results["Hunting_Rate"] *= 100

print(results.round(2))

The group with both wider flanges on average and a larger left–right flange height difference had the highest hunting rate in the table. About 83% of its 35 observations hunted, so the next step is to see how that pattern holds up across cars.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.offsetbox import AnnotationBbox, DrawingArea
from matplotlib.patches import Wedge

fig, ax = plt.subplots(figsize=(12, 7))

# XGBoost predicted hunting probability
contour = ax.contourf(
    W, H, Z,
    levels=np.linspace(0, 1, 21),
    cmap="RdYlGn_r",
    alpha=0.65
)

plt.colorbar(contour, ax=ax, label="Predicted Hunting Probability")

# Group actual observations by measurement combination
counts = final_data.groupby(
    ["Flange Width_Avg", "Flange Height (Sh)_Diff"]
)["Truck_Hunted"].agg(
    Total="count",
    Hunted="sum"
).reset_index()

counts = counts[
    counts["Flange Width_Avg"].between(28, 35) &
    counts["Flange Height (Sh)_Diff"].between(0, 2.5)
]

# Draw circular pie charts with fixed display size
for _, row in counts.iterrows():
    x = row["Flange Width_Avg"]
    y = row["Flange Height (Sh)_Diff"]
    fraction = row["Hunted"] / row["Total"]

    size = 16
    radius = size / 2

    drawing = DrawingArea(size, size, 0, 0)

    # Hunting portion (red)
    drawing.add_artist(Wedge(
        (radius, radius), radius,
        0, 360 * fraction,
        facecolor="red",
        edgecolor="black",
        linewidth=0.5
    ))

    # Non-hunting portion (blue)
    drawing.add_artist(Wedge(
        (radius, radius), radius,
        360 * fraction, 360,
        facecolor="blue",
        edgecolor="black",
        linewidth=0.5
    ))

    ax.add_artist(AnnotationBbox(
        drawing, (x, y),
        frameon=False,
        pad=0,
        zorder=5
    ))

# Candidate thresholds
ax.axvline(33.0, color="black", linestyle="--", linewidth=1.5)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1.5)

ax.set_xlim(28, 35)
ax.set_ylim(0, 2.5)

ax.set_aspect("auto")

ax.set_xlabel("Average Flange Width")
ax.set_ylabel("Absolute flange-height difference (L/R)")
ax.set_title("XGBoost model response and observed hunting fractions")

ax.legend(handles=[
    Patch(facecolor="red", label="Hunting"),
    Patch(facecolor="blue", label="No Hunting")
], loc="upper left")

plt.tight_layout()
plt.show()

The background color shows the hunting probability predicted by XGBoost for each combination of average flange width and left–right flange height difference. Each pie shows what actually happened at a measured combination. Red is the share that hunted and blue is the share that did not. The dashed lines mark the candidate cutoffs.

In [ ]:
# Calculate hunting rates for each observed measurement combination
combo_results = final_data.groupby(
    ["Flange Width_Avg", "Flange Height (Sh)_Diff"]
)["Truck_Hunted"].agg(
    Count="count",
    Hunting_Rate="mean"
).reset_index()

combo_results["Hunting_Rate"] *= 100

# Only include combinations with at least 5 observations
combo_results = combo_results[combo_results["Count"] >= 5]

# Rank combinations by hunting rate
combo_results = combo_results.sort_values(
    ["Hunting_Rate", "Count"],
    ascending=[False, False]
)

print(combo_results.head(15).round(3).to_string(index=False))

In [ ]:
# Check whether the top combinations occur across multiple cars
combo_support = final_data.groupby(
    ["Flange Width_Avg", "Flange Height (Sh)_Diff"]
).agg(
    Observations=("Truck_Hunted", "size"),
    Hunting_Rate=("Truck_Hunted", "mean"),
    Unique_Cars=("Car", "nunique")
).reset_index()

combo_support["Hunting_Rate"] *= 100

print(
    combo_support.sort_values(
        ["Hunting_Rate", "Unique_Cars"],
        ascending=False
    ).head(15).round(3).to_string(index=False)
)

In [ ]:
# Find high-hunting combinations supported by multiple cars
results = []

for width in [31, 31.5, 32, 32.5, 33]:
    for height in [0.5, 0.75, 1, 1.25, 1.5]:

        subset = final_data[
            (final_data["Flange Width_Avg"] >= width) &
            (final_data["Flange Height (Sh)_Diff"] >= height)
        ]

        if len(subset) >= 28 and subset["Car"].nunique() >= 4:
            results.append({
                "Min Width": width,
                "Min Height Diff": height,
                "Observations": len(subset),
                "Unique Cars": subset["Car"].nunique(),
                "Hunting Rate (%)": subset["Truck_Hunted"].mean() * 100
            })

threshold_results = pd.DataFrame(results)

print(
    threshold_results.sort_values(
        ["Hunting Rate (%)", "Unique Cars"],
        ascending=False
    ).head(10).round(2).to_string(index=False)
)

## Checking a selected subgroup on held-out cars

The search below selects a width/height rule using training cars, then checks
the selected rows of the omitted car. Its hunting fraction is conditional on
being selected, not classifier accuracy. The candidate grid was developed
during this analysis, so this is still exploratory.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
import pandas as pd
import numpy as np

widths = [31, 31.5, 32, 32.5, 33]
heights = [0.5, 0.75, 1, 1.25, 1.5]

logo = LeaveOneGroupOut()
results = []

for train_idx, test_idx in logo.split(final_data, groups=final_data["Car"]):
    train = final_data.iloc[train_idx]
    test = final_data.iloc[test_idx]

    candidates = []

    # Find the strongest combination using training cars only
    for width in widths:
        for height in heights:
            subset = train[
                (train["Flange Width_Avg"] >= width) &
                (train["Flange Height (Sh)_Diff"] >= height)
            ]

            if len(subset) >= 28 and subset["Car"].nunique() >= 4:
                candidates.append((
                    subset["Truck_Hunted"].mean(),
                    width,
                    height
                ))

    if not candidates:
        continue

    rate, best_width, best_height = max(candidates)

    # Evaluate the selected combination on the excluded car
    selected = test[
        (test["Flange Width_Avg"] >= best_width) &
        (test["Flange Height (Sh)_Diff"] >= best_height)
    ]

    results.append({
        "Car": test["Car"].iloc[0],
        "Width": best_width,
        "Height Diff": best_height,
        "Matched": len(selected),
        "Hunting Rate": selected["Truck_Hunted"].mean()
        if len(selected) > 0 else np.nan
    })

validation_results = pd.DataFrame(results)

print(validation_results.round(3).to_string(index=False))

print("\nMost common selected combinations:")
print(
    validation_results.groupby(["Width", "Height Diff"])
    .size()
    .sort_values(ascending=False)
)

print("\nTotal matched observations:",
      validation_results["Matched"].sum())

Each row shows the cutoffs picked using the other cars and what happened when those cutoffs were applied to the car in that row. For example, 6 of the 7 matches for car 4 hunted. When a car had zero matches, its hunting rate shows up as `NaN`.

The most common pair was **average flange width of at least 33.0 mm** and **left–right flange height difference of at least 0.50 mm**. It was picked in 12 of 14 rounds. In those 12 rounds, 34 of the 42 held-out observations that met both cutoffs hunted (81.0%).

In [ ]:
valid = validation_results.dropna(subset=["Hunting Rate"])

total_hunted = (
    valid["Matched"] * valid["Hunting Rate"]
).sum()

total_matched = valid["Matched"].sum()

validated_rate = total_hunted / total_matched

print(f"Held-car hunting fraction among selected rows: {validated_rate:.2%}")
print(f"Matched observations: {total_matched}")
print(f"Cars with matches: {len(valid)}")
print(f"Overall dataset hunting rate: {final_data['Truck_Hunted'].mean():.2%}")

When we held out one car at a time, the cutoffs picked using the other cars flagged 77 observations across eight held-out cars. About 77% of those observations hunted, compared with 64% across the full dataset.

In [ ]:
## What this supports

Flange summaries are useful leads for describing hunting, but the moderate
test performance and variation across held cars limit a predictive conclusion.
The subgroup summary covers only cars with qualifying rows; the printed
coverage must be read alongside its hunting fraction. Repeated rail rows do
not add independent vehicles. We use notebook 08 to compare model families
and shared wear summaries under one truck/rail, complete-car evaluation.